# Calculate product footprints and substance contributions

## Notebook README

**Related publication**

If you utilize any portions of the code, results, or draw inspiration for your projects, please give proper credit by citing the published article below.

 - Title: Residual biomass to bio-based chemicals and plastics: ex-ante screening methodology for prioritizing high-impact substitutions
 - Authors: [Nicolas LIENART](https://orcid.org/0009-0001-3259-2819), [Thibaut LECOMPTE](https://orcid.org/0000-0001-9237-8454), [Lorie HAMELIN](https://orcid.org/0000-0001-9092-1900) 
 - Journal: Resources, Conservation and Recycling (RCR) - Elsevier
 - Doi: #todo
 - Code author: Nicolas LIENART
 - Git repository (Forge INRAE): https://forge.inrae.fr/nicolas.lienart/screen-lca-paper-supplementary-code
 - Git repository (GitHub): https://github.com/nicolnt/screen-lca-paper-supplementary-code

**Description and details**

This code generates the hotspot figure (Figure 2) showing the import and production amount of chemicals and plastics in the EU27, their unit GHG footprint (cradle-to-gate), the total estimated footprint (cradle-to-gate) and the estimated end of life impacts. Refer to the aforementioned main manuscript and accompanying supplementary information documents for more details.

**Updates**

 - May 04, 2026: Ready for submission
 - August 11, 2026: Refactor code. Add support for multiple Ecoinvent database version. Split activities in multiple regions. Removed check in the function for substances LCA that could prevent further recursive calls despite behing possibly relevants.
 - August 12, 2026: Refactor code. Fixed an error in the end-of-life footprint calculation where C was not converted to CO2. Calculate footprint share due to imports and show in the figure.
 - August 14, 2026: Refactor code. Improve figure rendering for readability and bring additional informations.
 - August 14, 2026: Extract Ecoinvent database code to a common module.
 - August 15, 2026: Update figure output code for easier change of Ecoinvent database version.
 - August 16, 2026: Separated figure generation from footprint data calculation.
 - August 19, 2026: Update end of life calculation method to consider different fates, based on plastics.

**Package versions**

 - See [`environment.yml`](./environment.yml)

**Licence**

This work is licenced under the Creative Commons Attribution (CC-BY 4.0) public licence.

## Initialization

### Python imports

In [1]:
import bw2data as bd
import bw2io as bi
import pandas as pd
import numpy as np
import re
from importlib.metadata import version

17:55:36+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
print('bw2data:', version('bw2data'), "commit:", "532e73e8eb86668ba07a5378279ca33c74f76990")
print('bw2io:', version('bw2io'))
print('pandas:', version('pandas'))
print('numpy:', version('numpy'))

bw2data: 4.7 commit: 532e73e8eb86668ba07a5378279ca33c74f76990
bw2io: 0.9.17
pandas: 2.2.3
numpy: 2.4.3


In [3]:
# NOTE: Local import
import sys
sys.path.append("../")

from Python_utils import brightway_database

### Import some global variables from file

Your `.env` file is a collection of key-value pairs, separated by a `=` sign. It should contain these lines with the relevant values:

```bash
ecoinvent_username="replace_with your Ecoinvent's licence username"
ecoinvent_password="replace_with your Ecoinvent's licence password"
project_name="replace with project name"
```

We can then access this information with the dotenv and Python's built-in `os` libraries.

In [4]:
import os
from dotenv import load_dotenv  # To read the contents of the .env file

load_dotenv(override=True)

# NOTE: Get Ecoinvent account login details from a different file
ECOINVENT_USERNAME = os.getenv("ecoinvent_username")
ECOINVENT_PASSWORD = os.getenv("ecoinvent_password")

BW_PROJECT_NAME = os.getenv("project_name")

In [ ]:
# NOTE: Manually set Brightway current project
# PROJECT_NAME = "a-brightway-project"

### Manually set other global variables

In [ ]:
UNIT_EOL_FOOTPRINT_COLUMN_NAME = 'Unit impact of end-of-life (kg CO2/kg)'
UNIT_FOOTPRINT_WITHOUT_EOL_COLUMN_NAME = "Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg)"
UNIT_FOOTPRINT_WITH_EOL_COLUMN_NAME = "Unit GHG footprint with end-of-life (IPCC 2021 GWP100, kg CO2e/kg)"
TOTAL_FOOTPRINT_WITHOUT_EOL_COLUMN_NAME = "Total GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg)"
TOTAL_FOOTPRINT_WITH_EOL_COLUMN_NAME = "Total GHG footprint with end-of-life (IPCC 2021 GWP100, kg CO2e/kg)"
QUANTITY_COLUMN_NAME = "Quantity (Mt/yr)"
UNIT_BTX_FOOTPRINT_COLUMN_NAME = "Unit GHG footprint of BTX (IPCC 2021 GWP100, kg CO2e/kg)"
OUTPUT_PRODUCT_CODE_COLUMN_NAME = "Total footprint rank"
ECOINVENT_ACTIVITY_COLUMN_NAME = "Ecoinvent activity name or proxy"
UNIT_FOOTPRINT_LOCAL_PRODUCTION_COLUMN_NAME = "Unit GHG footprint of local production share (IPCC 2021 GWP100, kg CO2e/kg)"
LOCAL_PRODUCTION_SHARE_COLUMN_NAME = "Local production share of available quantity (%)"

In [32]:
# NOTE: Refer to supplementary materials file and "Article_data/Step 3 - PRODCOM results.xlsx" for details and references

ECOINVENT_PLASTIC_WASTE_CARBON_MASS_SHARE = 0.748
END_OF_LIFE_FATES = [
    {
        'fate_name': "Open-pit burning",
        'fate_share': 0.073654391,
        'ecoinvent_activity_name': "treatment of waste plastic, mixture, open burning",
        'ecoinvent_activity_location': "GLO",
    },
    {
        'fate_name': "Dumpsites",
        'fate_share': 0.09631728,
        'ecoinvent_activity_name': "treatment of waste plastic, mixture, unsanitary landfill, dry infiltration class (100mm)",
        'ecoinvent_activity_location': "GLO",
    },
    {
        'fate_name': "Landfilled",
        'fate_share': 0.492917847,
        'ecoinvent_activity_name': "treatment of waste plastic, mixture, sanitary landfill",
        'ecoinvent_activity_location': "RoW",
    },
    {
        'fate_name': "Incinerated",
        'fate_share': 0.1898017,
        'ecoinvent_activity_name': "treatment of waste plastic, mixture, municipal incineration",
        'ecoinvent_activity_location': "RoW",
    },
]

In [6]:
# NOTE: Name corresponds to Ecoinvent's product name
SUBSTANCE_LIST = [
    {
        "name":'propylene',
        "CAS":'115-07-1'
    },
    {
        "name":'ethylene',
        "CAS":'74-85-1'
    },
    {
        "name":'methanol',
        "CAS":'000067-56-1'
    },
    {
        "name":'xylene, mixed',
        "CAS":'',
        "type": "BTX"
    },
    {
        "name":'o-xylene',
        "CAS":'95-47-6',
        "type": "BTX"
    },
    {
        "name":'p-xylene',
        "CAS":'106-42-3',
        "type": "BTX"
    },
    {
        "name":'toluene, liquid',
        "CAS":'108-88-3',
        "type": "BTX"
    },
    {
        "name":'benzene',
        "CAS":'000071-43-2',
        "type": "BTX"
    }
]

In [7]:
ECOINVENT_DATABASE_NAMES = ['ecoinvent-3.11-cutoff', 'ecoinvent-3.12-cutoff']

In [8]:
ECOINVENT_DATABASES = {}

for ecoinvent_database_name in ECOINVENT_DATABASE_NAMES:
    # NOTE: Produces a string which looks something like "ecoinvent-x.x.x" out of "ecoinvent-x.x.x-model"
    method_key_db_name = "-".join(ecoinvent_database_name.split("-")[0:2])

    # NOTE: Select the relevant key, based on Ecoinvent version (no matter the system model)
    gwp_key = (method_key_db_name, "IPCC 2021", "climate change: total (excl. biogenic CO2)", "global warming potential (GWP100)")

    ECOINVENT_DATABASES[ecoinvent_database_name] = brightway_database.BrightwayDatabase(bw_database_name=ecoinvent_database_name, bw_project_name=BW_PROJECT_NAME, method_key=gwp_key)

# ECOINVENT_DATABASES = ecoinvent_databases.get_ecoinvent_databases(ECOINVENT_DATABASE_NAMES, project_name=BW_PROJECT_NAME)
ECOINVENT_DATABASES

{'ecoinvent-3.11-cutoff': <Python_utils.brightway_database.BrightwayDatabase at 0x7f74370a63c0>,
 'ecoinvent-3.12-cutoff': <Python_utils.brightway_database.BrightwayDatabase at 0x7f7437283d90>}

### Additional settings

In [9]:
pd.options.display.max_columns = 100
pd.options.display.max_rows = 5

## Setup Brightway project

In [ ]:
# NOTE: List Brightway projects
sorted(bd.projects)

In [49]:
# NOTE: Select the brightway project to use for this notebook
bd.projects.set_current(name=BW_PROJECT_NAME)
# bd.projects.set_current(name="template-ecoinvent-3.11-consequential")
bd.projects.current

'20260809-paper-1-screen-lca'

### Import Ecoinvent (required only once per project and Ecoinvent database version)

In [ ]:
# NOTE: Download and load ecoinvent into the current project. Takes several minutes to complete.
bi.import_ecoinvent_release(
    version='3.12',
    system_model='cutoff',
    username=ECOINVENT_USERNAME, 
    password=ECOINVENT_PASSWORD 
)

In [16]:
# NOTE: List available databases in the current Brightway project
bd.databases

Databases dictionary with 4 object(s):
	ecoinvent-3.11-biosphere
	ecoinvent-3.11-cutoff
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-cutoff

## General footprint calculation functions

In [50]:
def ecoinvent_region_and_share_series_to_df(ecoinvent_region_data):

    location_index = ecoinvent_region_data.loc[ecoinvent_region_data.index.str.endswith('location')].index.values

    # NOTE: Extract the part of the name that is common to use as index
    ecoinvent_region_index = [re.split(r"(^.*region \d)", s)[1] for s in location_index]            

    # NOTE: Construct the dataframe
    return pd.DataFrame([
            ecoinvent_region_data.loc[ecoinvent_region_data.index.str.endswith('location')].values,
            ecoinvent_region_data.loc[ecoinvent_region_data.index.str.endswith('share')].values
        ], index=['location', 'share'], columns=ecoinvent_region_index).T

In [ ]:
def lca_contribution_of_substance(activity_id, substance, recursive_call_left, bw_db, cutoff=1):

    activity = bd.get_activity(activity_id)
    
    # NOTE: Reached the end of the recursive path, don't go further.
    if recursive_call_left == 0:
        return 0
    
    else:
        activity_score = 0
        for exchange in activity.technosphere():

            exchange_id = exchange.input.id
            exchange_amount = exchange['amount']

            # NOTE: Assuming everything would be accounted as kg
            if exchange['unit'] != 'kilogram':
                continue

            
            # NOTE: Optimisation: Check if activity ISIC code is != than '19', '20', '21', '22'.
            # Assuming other type of activities won't deliver the substances.
            # See: https://ilostat.ilo.org/methods/concepts-and-definitions/classification-economic-activities/
            # 19 Manufacture of coke and refined petroleum products
            # 20 Manufacture of chemicals and chemical products
            # 21 Manufacture of basic pharmaceutical products and pharmaceutical preparations
            # 22 Manufacture of rubber and plastics products
            isic_code = ''
            if 'classifications' in exchange.input and len([e for e in exchange.input['classifications'] if e[0] == 'ISIC rev.4 ecoinvent']) > 0:
                isic_code = [e for e in exchange.input['classifications'] if e[0] == 'ISIC rev.4 ecoinvent'][0][1]
                # NOTE: Check the first two digits of ISIC classification and skip if other code is used
                if isic_code == '' or isic_code[0:2] not in ['19', '20', '21', '22']:
                    continue

            # NOTE: Optimisation: Apply a cutoff on the mass
            share = exchange['amount'] / activity['production amount']
            if share < cutoff:
                continue

            # NOTE: If CAS or name is matched, calculate and return footprint and stop there.
            if ('CAS number' in exchange and exchange['CAS number'] == substance['CAS']) \
                or exchange['name'].lower() == substance['name'].lower():
                activity_score += bw_db.calculate_LCA_optimized(activity_id=exchange_id, amount=exchange_amount)

            else:
                exchange_score = exchange_amount * lca_contribution_of_substance(activity_id=exchange_id,
                                                                                    substance=substance,
                                                                                    recursive_call_left=(recursive_call_left-1),
                                                                                    cutoff=cutoff * (1/share),
                                                                                    bw_db=bw_db)
                
                activity_score += exchange_score

        return activity_score

## Import Prodcom hotspot products for footprint calculation

In [52]:
hotspot_products_df = pd.read_csv('../Output_data/Hotspot products export - with ecoinvent regions.csv', index_col=0, dtype={
    'PRODCOM code': str,
    'HS22 code': str,
})

In [53]:
# NOTE: Remove unused columns from the CSV, remove listed columns below
hotspot_products_with_footprint_df = hotspot_products_df.loc[:, ~hotspot_products_df.columns.isin([
    "Carbon atoms",
    "Molecular weight (MW)",
    "Molecular formula (MF)",
    "HS22 code",
    "Note and references",
    "MF and MW reference"
    ])].copy()

In [54]:
hotspot_products_with_footprint_df

,PRODCOM code,Long name,Figure name,Quantity (kg/yr),Quantity (Mt/yr),Local production share of available quantity (%),Ecoinvent activity name or proxy,Carbon mass share,Ecoinvent 3.11 activity region 1 location,Ecoinvent 3.11 activity region 1 share,Ecoinvent 3.11 activity region 2 location,Ecoinvent 3.11 activity region 2 share,Ecoinvent 3.11 activity region 3 location,Ecoinvent 3.11 activity region 3 share,Ecoinvent 3.12 activity region 1 location,Ecoinvent 3.12 activity region 1 share,Ecoinvent 3.12 activity region 2 location,Ecoinvent 3.12 activity region 2 share,Ecoinvent 3.12 activity region 3 location,Ecoinvent 3.12 activity region 3 share,Ecoinvent 3.12 activity region 4 location,Ecoinvent 3.12 activity region 4 share,Ecoinvent 3.12 activity region 5 location,Ecoinvent 3.12 activity region 5 share
id,,,,,,,,,,,,,,,,,,,,,,,,
1,20165130,"Polypropylene, in primary forms",Polypropylene,1.105982e+10,11.059816,0.864021,"market for polypropylene, granulate",0.856277,GLO,1.00,NaN,NaN,NaN,NaN,RER,0.89,Asia without China,0.06,RoW,0.05,NaN,NaN,NaN,NaN
2,20141130,Ethylene,Ethylene,1.033749e+10,10.337494,0.879049,market for ethylene,0.856399,RER,0.96,RoW,0.04,NaN,NaN,RER w/o RU,0.94,TR,0.03,RU,0.02,US,0.01,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,20165450,"Polyamide -6,6 (PA 66)","Polyamide -6,6",1.089108e+09,1.089108,0.877484,market for nylon 6-6,0.642586,RER,0.91,RoW,0.09,NaN,NaN,RER,0.91,RoW,0.09,NaN,NaN,NaN,NaN,NaN,NaN
57,20165450,Polyamide -6 (PA 6),Polyamide -6,1.281304e+09,1.281304,0.877484,market for nylon 6,0.618274,RER,0.91,RoW,0.09,NaN,NaN,RER,0.91,RoW,0.09,NaN,NaN,NaN,NaN,NaN,NaN


## Unit footprint of products and substance contributions

In [ ]:
# NOTE: Took less than 10 s for first product (lca_score_cutoff_ratio = 0.01, iterations = 10)
# Took a few minutes (< 10 min) to run for two ecoinvent versions and 52 products 

lca_score_cutoff_ratio = 0.01 # Skip any flow contributing to less than x% of the product's footprint
iterations = 10

for ecoinvent_database_name in ECOINVENT_DATABASE_NAMES:

    ecoinvent_db: brightway_database.BrightwayDatabase = ECOINVENT_DATABASES[ecoinvent_database_name]

    # NOTE: Add constituents columns
    substances_column_names = [f"{substance['name']} - {ecoinvent_database_name}" for substance in SUBSTANCE_LIST]
    hotspot_products_with_footprint_df.loc[:, substances_column_names] = float(0)

    # NOTE: Add empty column to receive the impact scores
    unit_production_footprint_column_name = f"{UNIT_FOOTPRINT_WITHOUT_EOL_COLUMN_NAME} - {ecoinvent_database_name}"
    production_footprint_local_production_column_name = f"{UNIT_FOOTPRINT_LOCAL_PRODUCTION_COLUMN_NAME} - {ecoinvent_database_name}"
    hotspot_products_with_footprint_df[unit_production_footprint_column_name] = float(np.nan)

    ecoinvent_version_number = re.search(r"-([\d\.]+)-", ecoinvent_database_name).group(1)

    for hotspot_product_index, hotspot_product in hotspot_products_with_footprint_df.iterrows():
        ecoinvent_activity_name = hotspot_product[ECOINVENT_ACTIVITY_COLUMN_NAME]
        local_production_share = hotspot_product[LOCAL_PRODUCTION_SHARE_COLUMN_NAME]

        # NOTE: This product has a fixed impact value
        if ecoinvent_activity_name.startswith('manual:'):
            product_score = float(ecoinvent_activity_name.split(':')[1])
            footprint_local_production = product_score * local_production_share
        else:
            product_score = 0

            # NOTE: Get a list (series) of regions location and share for the current product
            ecoinvent_regions_data = hotspot_product.loc[
                    hotspot_product.index.str.startswith(f"Ecoinvent {ecoinvent_version_number} activity") &
                    ~hotspot_product.isna()
                ]

            activity_regions_df = ecoinvent_region_and_share_series_to_df(ecoinvent_regions_data)

            # NOTE: Loop through relevant regions for the current product
            for region_index, activity_region in activity_regions_df.iterrows():
                # NOTE: Get LCA result for the share of current region
                ecoinvent_activity = ecoinvent_db.get_activity_by_name_specific_location(ecoinvent_activity_name, location=activity_region["location"])

                unit_product_score = ecoinvent_db.calculate_LCA_optimized(activity_id=ecoinvent_activity.id)

                activity_region_share = activity_region["share"]
                # NOTE: Get the share of impact by applying the local production share over first region's production footprint
                if region_index.endswith("1"):
                    footprint_local_production = unit_product_score * local_production_share

                # NOTE: Apply the region's share on the score to reflect its contribution
                region_footprint = unit_product_score * activity_region_share
                product_score += region_footprint

                # NOTE: Calculate the share of substances using only one database version
                # if ecoinvent_version == 'ecoinvent-3.12-cutoff':
                for substance in SUBSTANCE_LIST:
                    substance_score = lca_contribution_of_substance(activity_id=ecoinvent_activity.id,
                                                                    substance=substance,
                                                                    # scaling_factor=1,
                                                                    # total_footprint=region_footprint,
                                                                    recursive_call_left=iterations,
                                                                    cutoff=lca_score_cutoff_ratio,
                                                                    bw_db=ecoinvent_db)

                    # NOTE: Substance impact is connected to region shares at this point, upstream it becomes dependent only on Ecoinvent modelling
                    hotspot_products_with_footprint_df.loc[hotspot_product_index, f"{substance['name']} - {ecoinvent_database_name}"] += substance_score * activity_region_share

        # NOTE: Resulting product score from the different regions involved
        hotspot_products_with_footprint_df.at[hotspot_product_index, unit_production_footprint_column_name] = product_score
        hotspot_products_with_footprint_df.at[hotspot_product_index, production_footprint_local_production_column_name] = footprint_local_production

        # if hotspot_product_index == 1:
        #     break

In [62]:
hotspot_products_with_footprint_df

,PRODCOM code,Long name,Figure name,Quantity (kg/yr),Quantity (Mt/yr),Local production share of available quantity (%),Ecoinvent activity name or proxy,Carbon mass share,Ecoinvent 3.11 activity region 1 location,Ecoinvent 3.11 activity region 1 share,Ecoinvent 3.11 activity region 2 location,Ecoinvent 3.11 activity region 2 share,Ecoinvent 3.11 activity region 3 location,Ecoinvent 3.11 activity region 3 share,Ecoinvent 3.12 activity region 1 location,Ecoinvent 3.12 activity region 1 share,Ecoinvent 3.12 activity region 2 location,Ecoinvent 3.12 activity region 2 share,Ecoinvent 3.12 activity region 3 location,Ecoinvent 3.12 activity region 3 share,Ecoinvent 3.12 activity region 4 location,Ecoinvent 3.12 activity region 4 share,Ecoinvent 3.12 activity region 5 location,Ecoinvent 3.12 activity region 5 share,propylene - ecoinvent-3.11-cutoff,ethylene - ecoinvent-3.11-cutoff,methanol - ecoinvent-3.11-cutoff,"xylene, mixed - ecoinvent-3.11-cutoff",o-xylene - ecoinvent-3.11-cutoff,p-xylene - ecoinvent-3.11-cutoff,"toluene, liquid - ecoinvent-3.11-cutoff",benzene - ecoinvent-3.11-cutoff,"Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff","Unit GHG footprint of local production share (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff",propylene - ecoinvent-3.12-cutoff,ethylene - ecoinvent-3.12-cutoff,methanol - ecoinvent-3.12-cutoff,"xylene, mixed - ecoinvent-3.12-cutoff",o-xylene - ecoinvent-3.12-cutoff,p-xylene - ecoinvent-3.12-cutoff,"toluene, liquid - ecoinvent-3.12-cutoff",benzene - ecoinvent-3.12-cutoff,"Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff","Unit GHG footprint of local production share (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff"
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,20165130,"Polypropylene, in primary forms",Polypropylene,1.105982e+10,11.059816,0.864021,"market for polypropylene, granulate",0.856277,GLO,1.00,NaN,NaN,NaN,NaN,RER,0.89,Asia without China,0.06,RoW,0.05,NaN,NaN,NaN,NaN,2.289804,0.087581,0.466132,0.0,0.0,0.0,0.0,0.0,3.049226,2.634595,1.84108,0.097479,0.0,0.0,0.0,0.0,0.0,0.0,2.341639,2.024023
2,20141130,Ethylene,Ethylene,1.033749e+10,10.337494,0.879049,market for ethylene,0.856399,RER,0.96,RoW,0.04,NaN,NaN,RER w/o RU,0.94,TR,0.03,RU,0.02,US,0.01,NaN,NaN,0.000000,1.822986,0.000000,0.0,0.0,0.0,0.0,0.0,1.823393,1.600745,0.00000,1.879038,0.0,0.0,0.0,0.0,0.0,0.0,1.879215,1.664083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,20165450,"Polyamide -6,6 (PA 66)","Polyamide -6,6",1.089108e+09,1.089108,0.877484,market for nylon 6-6,0.642586,RER,0.91,RoW,0.09,NaN,NaN,RER,0.91,RoW,0.09,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,8.258322,7.241229,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,8.258558,7.241449
57,20165450,Polyamide -6 (PA 6),Polyamide -6,1.281304e+09,1.281304,0.877484,market for nylon 6,0.618274,RER,0.91,RoW,0.09,NaN,NaN,RER,0.91,RoW,0.09,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,9.312870,8.166357,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,9.313357,8.166795


In [ ]:
hotspot_products_with_footprint_df.to_csv('../Output_data/Hotspot products with footprint - intermediate.csv')

## Group similar products

In [39]:
# NOTE: Import from already exported previous result
hotspot_products_with_footprint_grouped_df = pd.read_csv('../Output_data/Hotspot products with footprint - intermediate.csv', index_col=0, dtype={
    'PRODCOM code': str,
    'HS22 code': str,
})

# NOTE: Remove unused columns from the CSV
hotspot_products_with_footprint_grouped_df = hotspot_products_with_footprint_grouped_df.loc[:, ~hotspot_products_with_footprint_grouped_df.columns.isin(
    [column_name for column_name in hotspot_products_with_footprint_grouped_df.columns.values if re.match(r"Ecoinvent [\d\.]+ activity region.*", column_name)]
    +
    [
        'Quantity (kg/yr)',
        'Ecoinvent activity name or proxy',
        'PRODCOM code'
    ]
)].copy()

In [40]:
hotspot_products_with_footprint_grouped_df

,Long name,Figure name,Quantity (Mt/yr),Local production share of available quantity (%),Carbon mass share,propylene - ecoinvent-3.11-cutoff,ethylene - ecoinvent-3.11-cutoff,methanol - ecoinvent-3.11-cutoff,"xylene, mixed - ecoinvent-3.11-cutoff",o-xylene - ecoinvent-3.11-cutoff,p-xylene - ecoinvent-3.11-cutoff,"toluene, liquid - ecoinvent-3.11-cutoff",benzene - ecoinvent-3.11-cutoff,"Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff","Unit GHG footprint of local production share (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff",propylene - ecoinvent-3.12-cutoff,ethylene - ecoinvent-3.12-cutoff,methanol - ecoinvent-3.12-cutoff,"xylene, mixed - ecoinvent-3.12-cutoff",o-xylene - ecoinvent-3.12-cutoff,p-xylene - ecoinvent-3.12-cutoff,"toluene, liquid - ecoinvent-3.12-cutoff",benzene - ecoinvent-3.12-cutoff,"Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff","Unit GHG footprint of local production share (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff"
id,,,,,,,,,,,,,,,,,,,,,,,,,
1,"Polypropylene, in primary forms",Polypropylene,11.059816,0.864021,0.856277,2.289804,0.087581,0.466132,0.0,0.0,0.0,0.0,0.0,3.049226,2.634595,1.84108,0.097479,0.0,0.0,0.0,0.0,0.0,0.0,2.341639,2.024023
2,Ethylene,Ethylene,10.337494,0.879049,0.856399,0.000000,1.822986,0.000000,0.0,0.0,0.0,0.0,0.0,1.823393,1.600745,0.00000,1.879038,0.0,0.0,0.0,0.0,0.0,0.0,1.879215,1.664083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,"Polyamide -6,6 (PA 66)","Polyamide -6,6",1.089108,0.877484,0.642586,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,8.258322,7.241229,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,8.258558,7.241449
57,Polyamide -6 (PA 6),Polyamide -6,1.281304,0.877484,0.618274,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,9.312870,8.166357,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,9.313357,8.166795


In [41]:
# NOTE: Perform complex aggregation involving more than one column at once (i.e., weighted average requires current column and weights refering to another datafram column)
def aggregate_output_columns(df, average_columns, weight_column):
    # NOTE: Compute weighted average using the product quantities
    average_dict = {
            column: lambda x: np.average(x, weights=df[weight_column]) for column in average_columns
        }

    # NOTE: Compute simple sum of product quantities
    sum_dict = {weight_column: "sum"}

    agg_df = df.aggregate(average_dict | sum_dict)
    return agg_df

In [42]:
# NOTE: Identify duplicated rows (i.e., same figure name)
duplicated_product_rows_bool = hotspot_products_with_footprint_grouped_df['Figure name'].duplicated(keep=False)

# NOTE: Select columns which are floats for which to compute average (except quantity which is simply summed)
float_data_columns = hotspot_products_with_footprint_grouped_df.select_dtypes('float').drop([QUANTITY_COLUMN_NAME], axis=1).columns.values

# NOTE: Call custom aggregation function
duplicated_rows_grouped_df = hotspot_products_with_footprint_grouped_df[duplicated_product_rows_bool]\
    .groupby(by=['Figure name'], as_index=False)\
        .apply(aggregate_output_columns,
               include_groups=False,
               average_columns=float_data_columns,
               weight_column=QUANTITY_COLUMN_NAME)

# NOTE: Select non duplicate rows
non_duplicated_rows_df = hotspot_products_with_footprint_grouped_df[~duplicated_product_rows_bool]

# NOTE: Merge non duplicate rows with the groupped duplicate rows
hotspot_products_with_footprint_grouped_df = pd.concat([non_duplicated_rows_df, duplicated_rows_grouped_df])

hotspot_products_with_footprint_grouped_df.reset_index(drop=True, inplace=True)

In [43]:
hotspot_products_with_footprint_grouped_df

,Long name,Figure name,Quantity (Mt/yr),Local production share of available quantity (%),Carbon mass share,propylene - ecoinvent-3.11-cutoff,ethylene - ecoinvent-3.11-cutoff,methanol - ecoinvent-3.11-cutoff,"xylene, mixed - ecoinvent-3.11-cutoff",o-xylene - ecoinvent-3.11-cutoff,p-xylene - ecoinvent-3.11-cutoff,"toluene, liquid - ecoinvent-3.11-cutoff",benzene - ecoinvent-3.11-cutoff,"Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff","Unit GHG footprint of local production share (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff",propylene - ecoinvent-3.12-cutoff,ethylene - ecoinvent-3.12-cutoff,methanol - ecoinvent-3.12-cutoff,"xylene, mixed - ecoinvent-3.12-cutoff",o-xylene - ecoinvent-3.12-cutoff,p-xylene - ecoinvent-3.12-cutoff,"toluene, liquid - ecoinvent-3.12-cutoff",benzene - ecoinvent-3.12-cutoff,"Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff","Unit GHG footprint of local production share (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff"
0,"Polypropylene, in primary forms",Polypropylene,11.059816,0.864021,0.856277,2.289804,0.087581,0.466132,0.0,0.0,0.0,0.0,0.0,3.049226,2.634595,1.84108,0.097479,0.000000,0.0,0.0,0.0,0.0,0.0,2.341639,2.024023
1,Ethylene,Ethylene,10.337494,0.879049,0.856399,0.000000,1.822986,0.000000,0.0,0.0,0.0,0.0,0.0,1.823393,1.600745,0.00000,1.879038,0.000000,0.0,0.0,0.0,0.0,0.0,1.879215,1.664083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49,NaN,Polyvinyl acetate,1.165636,0.943691,0.558068,0.000000,0.608155,0.542041,0.0,0.0,0.0,0.0,0.0,3.766482,3.531682,0.00000,0.628084,0.491587,0.0,0.0,0.0,0.0,0.0,3.792950,3.557942
50,NaN,Polyvinyl chloride,5.798768,0.898553,0.384352,0.000000,0.774747,0.000000,0.0,0.0,0.0,0.0,0.0,2.389817,2.105985,0.00000,0.808051,0.000000,0.0,0.0,0.0,0.0,0.0,2.447081,2.168123


## Calculate the end of life footprint

In [44]:
# ECOINVENT_PLASTIC_WASTE_CARBON_MASS_SHARE

def calculate_eol_unit_footprint(carbon_mass, ecoinvent_version):
    total_eol_score = 0
    for fate in END_OF_LIFE_FATES:
        ecoinvent_db: brightway_database.BrightwayDatabase = ECOINVENT_DATABASES[ecoinvent_version]
        activity = ecoinvent_db.get_activity_by_name_specific_location(name=fate['ecoinvent_activity_name'], location=fate['ecoinvent_activity_location'])
        carbon_scale_factor = (-1) / ECOINVENT_PLASTIC_WASTE_CARBON_MASS_SHARE
        fate_score = ecoinvent_db.calculate_LCA(activity_id=activity.id, amount=carbon_scale_factor)
        real_fate_score = fate_score * fate['fate_share'] * carbon_mass
        total_eol_score += real_fate_score
        # print(f"Fate: {fate['fate_name']}. Score = {real_fate_score}")

    return total_eol_score

In [ ]:
# NOTE: Calculate GHG unit footprint of end-of-life value
# Takes less than 15s
for ecoinvent_database_name in ECOINVENT_DATABASE_NAMES:

    unit_eol_footprint_column_name = f"{UNIT_EOL_FOOTPRINT_COLUMN_NAME} - {ecoinvent_database_name}"
    hotspot_products_with_footprint_grouped_df[unit_eol_footprint_column_name] = float('nan')

    for hotspot_product_index, hotspot_product in hotspot_products_with_footprint_grouped_df.iterrows():
        
        # NOTE: %C (%mol wt)
        carbon_mass_share = hotspot_product['Carbon mass share']
        eol_score = calculate_eol_unit_footprint(carbon_mass=carbon_mass_share, ecoinvent_version=ecoinvent_database_name)

        # print(f"Product: {hotspot_product["Figure name"]}")
        # print(f"Carbon: {carbon_mass_share:.1%}")
        # print(f"Production score: {hotspot_product["Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff"]:.2} kgCO2-eq/kg")
        # print(f"EoL score: {eol_score:.2} kgCO2-eq/kg")

        # NOTE: g CO2/g molecule
        hotspot_products_with_footprint_grouped_df.at[hotspot_product_index, unit_eol_footprint_column_name] = eol_score

## Calculate total footprints

In [46]:
for ecoinvent_database_name in ECOINVENT_DATABASE_NAMES:
    # NOTE: Add empty columns to receive the impact scores
    unit_eol_footprint_column_name = f"{UNIT_EOL_FOOTPRINT_COLUMN_NAME} - {ecoinvent_database_name}"
    unit_footprint_with_eol_column_name = f"{UNIT_FOOTPRINT_WITH_EOL_COLUMN_NAME} - {ecoinvent_database_name}"
    total_footprint_without_eol_column_name = f"{TOTAL_FOOTPRINT_WITHOUT_EOL_COLUMN_NAME} - {ecoinvent_database_name}"
    total_footprint_column_name = f"{TOTAL_FOOTPRINT_WITH_EOL_COLUMN_NAME} - {ecoinvent_database_name}"

    hotspot_products_with_footprint_grouped_df[[unit_footprint_with_eol_column_name, total_footprint_without_eol_column_name, total_footprint_column_name]] = float('nan')

    unit_footprint_without_eol_column_name = f"{UNIT_FOOTPRINT_WITHOUT_EOL_COLUMN_NAME} - {ecoinvent_database_name}"

    for hotspot_product_index, hotspot_product in hotspot_products_with_footprint_grouped_df.iterrows():

        # NOTE: Calculate Unit GHG impact with end-of-life
        unit_footprint_with_eol = hotspot_product[unit_footprint_without_eol_column_name] + hotspot_product[unit_eol_footprint_column_name]
        hotspot_products_with_footprint_grouped_df.at[hotspot_product_index, unit_footprint_with_eol_column_name] = unit_footprint_with_eol

        product_quantity = hotspot_product[QUANTITY_COLUMN_NAME]

        # NOTE: Calculate total value without end-of-life
        total_footprint_without_eol = hotspot_product[unit_footprint_without_eol_column_name] * product_quantity
        hotspot_products_with_footprint_grouped_df.at[hotspot_product_index, total_footprint_without_eol_column_name] = total_footprint_without_eol

        # NOTE: Calculate total value with end-of-life
        total_footprint = unit_footprint_with_eol * product_quantity
        hotspot_products_with_footprint_grouped_df.at[hotspot_product_index, total_footprint_column_name] = total_footprint

## Sum BTX values

In [47]:
# NOTE: Calculate unit BTX footprint value (sum of the different BTX components)

for ecoinvent_database_name in ECOINVENT_DATABASE_NAMES:

    # NOTE: Add constituents columns
    substances_column_names = [f"{substance['name']} - {ecoinvent_database_name}" for substance in SUBSTANCE_LIST if substance.get('type') == "BTX"]
    btx_total_column_name = f"{UNIT_BTX_FOOTPRINT_COLUMN_NAME} - {ecoinvent_database_name}"

    hotspot_products_with_footprint_grouped_df[btx_total_column_name] = float('nan')

    for hotspot_product_index, hotspot_product in hotspot_products_with_footprint_grouped_df.iterrows():
        btx_total = hotspot_product[substances_column_names].sum()
        hotspot_products_with_footprint_grouped_df.at[hotspot_product_index, btx_total_column_name] = btx_total

In [48]:
hotspot_products_with_footprint_grouped_df

,Long name,Figure name,Quantity (Mt/yr),Local production share of available quantity (%),Carbon mass share,propylene - ecoinvent-3.11-cutoff,ethylene - ecoinvent-3.11-cutoff,methanol - ecoinvent-3.11-cutoff,"xylene, mixed - ecoinvent-3.11-cutoff",o-xylene - ecoinvent-3.11-cutoff,p-xylene - ecoinvent-3.11-cutoff,"toluene, liquid - ecoinvent-3.11-cutoff",benzene - ecoinvent-3.11-cutoff,"Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff","Unit GHG footprint of local production share (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff",propylene - ecoinvent-3.12-cutoff,ethylene - ecoinvent-3.12-cutoff,methanol - ecoinvent-3.12-cutoff,"xylene, mixed - ecoinvent-3.12-cutoff",o-xylene - ecoinvent-3.12-cutoff,p-xylene - ecoinvent-3.12-cutoff,"toluene, liquid - ecoinvent-3.12-cutoff",benzene - ecoinvent-3.12-cutoff,"Unit GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff","Unit GHG footprint of local production share (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff",Unit impact of end-of-life (kg CO2/kg) - ecoinvent-3.11-cutoff,Unit impact of end-of-life (kg CO2/kg) - ecoinvent-3.12-cutoff,"Unit GHG footprint with end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff","Total GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff","Total GHG footprint with end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff","Unit GHG footprint with end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff","Total GHG footprint without end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff","Total GHG footprint with end-of-life (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff","Unit GHG footprint of BTX (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.11-cutoff","Unit GHG footprint of BTX (IPCC 2021 GWP100, kg CO2e/kg) - ecoinvent-3.12-cutoff"
0,"Polypropylene, in primary forms",Polypropylene,11.059816,0.864021,0.856277,2.289804,0.087581,0.466132,0.0,0.0,0.0,0.0,0.0,3.049226,2.634595,1.84108,0.097479,0.000000,0.0,0.0,0.0,0.0,0.0,2.341639,2.024023,0.791465,0.791418,3.840691,33.723875,42.477338,3.133057,25.898093,34.651032,0.0,0.0
1,Ethylene,Ethylene,10.337494,0.879049,0.856399,0.000000,1.822986,0.000000,0.0,0.0,0.0,0.0,0.0,1.823393,1.600745,0.00000,1.879038,0.000000,0.0,0.0,0.0,0.0,0.0,1.879215,1.664083,0.791578,0.791531,2.614972,18.849318,27.032255,2.670746,19.426374,27.608820,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49,NaN,Polyvinyl acetate,1.165636,0.943691,0.558068,0.000000,0.608155,0.542041,0.0,0.0,0.0,0.0,0.0,3.766482,3.531682,0.00000,0.628084,0.491587,0.0,0.0,0.0,0.0,0.0,3.792950,3.557942,0.515828,0.515797,4.282310,4.390346,4.991614,4.308748,4.421199,5.022431,0.0,0.0
50,NaN,Polyvinyl chloride,5.798768,0.898553,0.384352,0.000000,0.774747,0.000000,0.0,0.0,0.0,0.0,0.0,2.389817,2.105985,0.00000,0.808051,0.000000,0.0,0.0,0.0,0.0,0.0,2.447081,2.168123,0.355260,0.355239,2.745077,13.857995,15.918068,2.802320,14.190056,16.250005,0.0,0.0


## Final export

In [49]:
hotspot_products_with_footprint_grouped_df.to_csv('../Output_data/Hotspot products with footprint - processed.csv', index=None)